<a href="https://colab.research.google.com/github/Chetansahney/projects/blob/main/LLM_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT Language Model from Scratch

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True);
    loss.backward()
    optimizer.step()

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

10.788929 M parameters
step 0: train loss 4.2221, val loss 4.2306
step 500: train loss 1.7580, val loss 1.9142
step 1000: train loss 1.3913, val loss 1.6009
step 1500: train loss 1.2667, val loss 1.5269
step 2000: train loss 1.1859, val loss 1.4966
step 2500: train loss 1.1197, val loss 1.4834
step 3000: train loss 1.0709, val loss 1.4803
step 3500: train loss 1.0209, val loss 1.5030
step 4000: train loss 0.9597, val loss 1.5095
step 4500: train loss 0.9124, val loss 1.5345
step 4999: train loss 0.8540, val loss 1.5477

But with presorts of that ever I was whim
My worst Capitol,--like a fool,--'
To learn that trouble too: if to be left, I wot;
For that of this fearful wretch, it standing to me:
O, when my sons are or more than they in right;
More than 'twill'd your own then bent it,
Yet she dares: if your son, to flesh it there, your
Flesh plightily cries and puck, 'tis not the pack that a vergin.
Thus back'd willingly, not one that lives;
So my unwies impressing hours in his time!

SL

This notebook implements a Generative Pre-trained Transformer (GPT) language model from scratch using PyTorch. The model is trained on a text file (`input.txt`) to predict the next character in a sequence.

## Hyperparameters

These are the configuration settings for the model and training process:

*   `batch_size`: Number of independent sequences processed in parallel.
*   `block_size`: Maximum context length for predictions.
*   `max_iters`: Total number of training iterations.
*   `eval_interval`: How often to evaluate the model's loss on train and validation sets.
*   `learning_rate`: Learning rate for the AdamW optimizer.
*   `device`: Specifies whether to use a GPU (`cuda`) or CPU for training.
*   `eval_iters`: Number of batches to use for evaluating loss.
*   `n_embd`: Dimensionality of the token and position embeddings.
*   `n_head`: Number of attention heads in the MultiHeadAttention mechanism.
*   `n_layer`: Number of transformer blocks.
*   `dropout`: Dropout rate applied in various layers to prevent overfitting.

## Data Preparation

1.  **Load Text Data**: Reads the `input.txt` file (e.g., Tiny Shakespeare dataset).
2.  **Vocabulary Creation**: Extracts all unique characters from the text to form a vocabulary.
3.  **Encoder/Decoder**: Creates mappings (`stoi`, `itos`) between characters and integers for encoding and decoding text.
4.  **Train/Validation Split**: Divides the encoded data into training (90%) and validation (10%) sets.
5.  **`get_batch` Function**: A utility function to generate batches of input (`x`) and target (`y`) sequences for training and evaluation. Each `x` is a sequence of `block_size` characters, and `y` contains the next `block_size` characters (shifted by one position).

## Model Architecture

### `Head` (Self-Attention Head)

Implements a single self-attention head. It computes queries, keys, and values from the input, calculates attention scores, applies masking for causality (ensuring a token can only attend to previous tokens), and then performs a weighted aggregation of values.

*   `key`, `query`, `value`: Linear layers to project input into key, query, and value spaces.
*   `tril`: A lower triangular matrix used for masking future tokens.
*   `dropout`: Prevents overfitting by randomly setting a fraction of outputs to zero.

### `MultiHeadAttention`

Combines multiple `Head` instances in parallel. The outputs from individual heads are concatenated and then passed through a linear projection layer.

*   `heads`: A list of `Head` modules.
*   `proj`: A linear layer to project the concatenated output back to the embedding dimension.

### `FeedFoward`

A simple two-layer neural network with a ReLU activation and dropout, applied to each token independently. This provides the computational aspect of the transformer block after the communication (attention).

### `Block` (Transformer Block)

Represents a single transformer block, consisting of a multi-head self-attention layer followed by a feed-forward network. Layer normalization is applied before both the self-attention and feed-forward layers, and residual connections are used.

*   `sa`: Multi-head self-attention module.
*   `ffwd`: Feed-forward network.
*   `ln1`, `ln2`: Layer normalization layers.

### `GPTLanguageModel`

The main GPT model. It comprises token embeddings, positional embeddings, a stack of transformer `Block`s, a final layer normalization, and a linear head for predicting logits.

*   `token_embedding_table`: Maps input token IDs to dense embedding vectors.
*   `position_embedding_table`: Maps token positions to dense embedding vectors, capturing sequence order.
*   `blocks`: A sequential stack of `Block` modules.
*   `ln_f`: Final layer normalization.
*   `lm_head`: A linear layer that projects the final block output to the vocabulary size, producing logits for the next token.
*   `_init_weights`: Custom weight initialization for better training stability.

## Training and Inference

### Training Loop

1.  **Model Instantiation**: Creates an instance of `GPTLanguageModel` and moves it to the specified `device` (GPU/CPU).
2.  **Optimizer**: Initializes the AdamW optimizer with the model parameters and learning rate.
3.  **Iteration**: The model trains for `max_iters` iterations.
    *   **Loss Estimation**: Every `eval_interval` steps, `estimate_loss()` is called to calculate and print the average training and validation losses.
    *   **Batch Sampling**: A batch of `(xb, yb)` is obtained using `get_batch('train')`.
    *   **Forward Pass**: The model performs a forward pass to get `logits` and `loss`.
    *   **Backward Pass**: Computes gradients (`loss.backward()`).
    *   **Optimizer Step**: Updates model parameters (`optimizer.step()`).
    *   **Zero Gradients**: Clears gradients before the next iteration (`optimizer.zero_grad()`).

### `generate` Function (Inference)

This method generates new text given a starting context.

1.  **Input**: Takes an initial `idx` (a tensor representing the starting token(s)) and `max_new_tokens` (number of tokens to generate).
2.  **Loop**: For each new token to be generated:
    *   **Crop Context**: The input context (`idx`) is cropped to `block_size` to ensure it doesn't exceed the model's context window.
    *   **Prediction**: The model performs a forward pass on the cropped context to get `logits`.
    *   **Focus on Last Timestep**: Only the logits for the last token in the sequence are considered, as we are predicting the *next* token.
    *   **Softmax**: Converts logits into probabilities.
    *   **Sampling**: A new token is sampled from the probability distribution using `torch.multinomial`.
    *   **Append**: The sampled token is appended to the `idx`, extending the context for the next prediction.

# README

## GPT from Scratch

This project implements a character-level Generative Pre-trained Transformer (GPT) language model using PyTorch, inspired by Andrej Karpathy's 'makemore' series. The model learns to predict the next character in a sequence based on the input text.

### Features

*   **Character-level Tokenization**: The model operates directly on characters, creating its vocabulary from the input text.
*   **Self-Attention Mechanism**: Utilizes multi-head self-attention to capture long-range dependencies in the text.
*   **Positional Embeddings**: Incorporates positional information to understand the order of tokens.
*   **Transformer Blocks**: Stacks multiple transformer encoder blocks, each with self-attention and a feed-forward network.
*   **Causal Masking**: Ensures that the model can only attend to previous tokens, making it suitable for language generation.
*   **Training Loop**: Includes a basic training loop with AdamW optimizer and loss estimation.
*   **Text Generation**: Capable of generating new text sequences based on a learned probability distribution over characters.

### How to Run

1.  **Prepare Data**: Make sure you have an `input.txt` file in the same directory as the notebook. You can download the Tiny Shakespeare dataset using the provided `wget` command in the notebook.
    ```bash
    wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
    ```
2.  **Execute Cells**: Run all the code cells in the notebook sequentially. The training process will start, and loss values will be printed periodically.
3.  **Generate Text**: After training completes, the model will generate a sample text output, demonstrating its learned language patterns.